# The Probability Density Function (PDF)

This notebook formalizes the curve from the previous notebook: the **probability density function**, the continuous counterpart of the PMF. Probabilities of intervals are **areas under the PDF**, and single points have area — and probability — zero. It is based on the seventh video of Lesson 2 (Probability Distributions).


## 1. Introduction

For discrete distributions each event had a probability: "3 heads in 10 tosses" is a number. For continuous ones, $P(\text{call} = \text{exactly 2 minutes}) = 0$ — but interval probabilities like $P(2 \leq X \leq 3)$ are perfectly meaningful. All of them are encoded in one function: the **PDF**.

After completing this notebook you will be able to:

- Compute interval probabilities as areas — starting with the uniform case, where they're just rectangles.
- State the three requirements a function must satisfy to be a PDF.
- Contrast the PMF (heights are probabilities) with the PDF (areas are probabilities).


## 2. Theory

### The uniform warm-up: heights vs. areas

Suppose the call center answers with equal probability at any time between 0 and 5 minutes. Split $[0, 5]$ into five 1-minute rectangles: each carries probability $\frac{1}{5} = 0.2$, so $P(2 \leq X \leq 3) = 0.2$.

Now split into ten 30-second intervals: $P(2 \leq X \leq 2.5) = \frac{1}{10} = 0.1$. **The heights didn't change — the widths halved, so the probabilities halved.** Probability is not the height of a tower; it is the **area** of the rectangle: height × width.

### Probabilities are areas

For a general (non-uniform) distribution the same logic applies with a curve:

$$
P(a \leq X \leq b) = \text{area under the PDF between } a \text{ and } b
$$

The probability of *exactly* 2 minutes is the "area" of a line segment — which is zero. That is the geometric reason single values have probability zero.

### The PDF and its requirements

The curve is the **probability density function**, written $f$ (or $f_X$ for clarity) — the continuous equivalent of the PMF's lowercase $p$. It represents **the rate at which probability accumulates around each point**. A function is a valid PDF if:

1. it is **defined for all real numbers** (it may be 0 over large regions — e.g. before 0 or beyond a cutoff — but it doesn't have to be: it can be positive everywhere as long as the tails shrink fast enough);
2. $f(x) \geq 0$ everywhere — negative values would mean negative probabilities;
3. the **total area under the curve is 1** — the probability of all possible outcomes.

### Summary: PMF vs. PDF

| | Discrete | Continuous |
|---|---|---|
| Values | finite or countable list | an interval |
| Function | PMF $p(x) = P(X = x)$ | PDF $f(x)$ |
| $P(X = x)$ | can be positive | always 0 |
| Probabilities from | **heights** | **areas** |
| Total | $\sum_x p(x) = 1$ | area under $f$ = 1 |


## 3. Worked Examples

### Example 1: The uniform call center

The PDF is flat: $f(x) = \frac{1}{5}$ on $[0, 5]$ and 0 elsewhere. (Height $\frac{1}{5}$ is forced: a $5 \times \frac{1}{5}$ rectangle has area 1.)

- $P(2 \leq X \leq 3) = 1 \times \frac{1}{5} = 0.2$
- $P(2 \leq X \leq 2.5) = 0.5 \times \frac{1}{5} = 0.1$
- $P(X = 2) = 0 \times \frac{1}{5} = 0$

### Example 2: The non-uniform call center

With the realistic curve (most calls 1-3 minutes), $P(1 \leq X \leq 2)$ is the area of that slice — larger than the slice from 4 to 5, because the curve is higher there. Same widths, different areas: the *density* varies.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)


### The uniform PDF and its rectangle probabilities


In [ ]:
def uniform_pdf(x, a=0, b=5):
    return np.where((x >= a) & (x <= b), 1 / (b - a), 0.0)

def uniform_interval_prob(lo, hi, a=0, b=5):
    lo, hi = max(lo, a), min(hi, b)
    return max(hi - lo, 0) * 1 / (b - a)

print(f"P(2 <= X <= 3)   = {uniform_interval_prob(2, 3):.2f}")
print(f"P(2 <= X <= 2.5) = {uniform_interval_prob(2, 2.5):.2f}")
print(f"P(X = 2)         = {uniform_interval_prob(2, 2):.2f}")
print(f"Total area       = {uniform_interval_prob(0, 5):.2f}")


### Checking the three PDF requirements numerically


In [ ]:
xs = np.linspace(-2, 7, 2_000)

candidates = {
    "uniform on [0,5]":      uniform_pdf(xs),
    "negative dip (invalid)": uniform_pdf(xs) - 0.05,
    "area 2 (invalid)":      2 * uniform_pdf(xs),
}

for name, fx in candidates.items():
    nonneg = bool((fx >= 0).all())
    area = np.trapz(fx, xs)
    verdict = "valid PDF" if nonneg and np.isclose(area, 1, atol=0.01) else "NOT a PDF"
    print(f"{name:25s} non-negative: {nonneg!s:5}  area: {area:5.2f}  -> {verdict}")


### Simulation: uniform call times


In [ ]:
calls = rng.uniform(0, 5, size=500_000)

print(f"Simulated P(2 <= X <= 3):   {((calls >= 2) & (calls <= 3)).mean():.4f}")
print(f"Simulated P(2 <= X <= 2.5): {((calls >= 2) & (calls <= 2.5)).mean():.4f}")
print(f"Simulated P(X == 2):        {(calls == 2.0).mean():.4f}")


## 5. Visualizations

### The video's halving experiment

Same height, half the width → half the probability. Probability is area, not height.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for ax, width, highlight, title in [
    (axes[0], 1.0, (2, 3), "five 1-min windows: P(2-3) = 0.2"),
    (axes[1], 0.5, (2, 2.5), "ten 30-sec windows: P(2-2.5) = 0.1"),
]:
    edges = np.arange(0, 5 + width, width)
    for e in edges[:-1]:
        selected = highlight[0] <= e < highlight[1]
        ax.add_patch(plt.Rectangle((e, 0), width, 0.2,
                                   facecolor="#e15759" if selected else "#9ecae1",
                                   edgecolor="white", linewidth=1.5))
    ax.set_xlim(-0.3, 5.3)
    ax.set_ylim(0, 0.30)
    ax.set_title(title)
    ax.set_xlabel("Call duration (min)")
    ax.axhline(0.2, color="gray", linestyle=":", linewidth=1)
    ax.text(5.05, 0.205, "height\nunchanged", fontsize=8, color="gray")

axes[0].set_ylabel("f(x) — density")
plt.suptitle("Uniform PDF: heights stay at 1/5; only the AREA changes with the window")
plt.tight_layout()
plt.show()


### Areas on a non-uniform PDF

The same call-center curve as the previous notebook, now read as a PDF: two equal-width windows with very different probabilities, and a zero-width 'window' at exactly 2 minutes.


In [ ]:
# Gamma-shaped density curve (same as the previous notebook), drawn exactly
xs = np.linspace(0, 5, 500)
shape, scale = 4.0, 0.55
pdf = xs ** (shape - 1) * np.exp(-xs / scale)
pdf /= np.trapz(pdf, xs)                    # normalize area to 1 on [0, 5]

plt.figure(figsize=(9, 5))
plt.plot(xs, pdf, color="#2c5f8a", linewidth=2, label="PDF f(x)")

m12 = (xs >= 1) & (xs <= 2)
m45 = (xs >= 4) & (xs <= 5)
plt.fill_between(xs[m12], pdf[m12], color="#e15759", alpha=0.75,
                 label=f"P(1≤X≤2) = {np.trapz(pdf[m12], xs[m12]):.2f}")
plt.fill_between(xs[m45], pdf[m45], color="#f1ce63", alpha=0.85,
                 label=f"P(4≤X≤5) = {np.trapz(pdf[m45], xs[m45]):.2f}")
plt.plot([2, 2], [0, np.interp(2, xs, pdf)], color="black", linewidth=2.5,
         label="P(X=2): a line segment, area 0")

plt.title("Probabilities are areas under the PDF — equal widths, unequal areas")
plt.xlabel("Call duration (min)")
plt.ylabel("f(x)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 6. Exercises

### Basic

1. A bus arrives uniformly at random within a 10-minute window: $f(x) = \frac{1}{10}$ on $[0, 10]$. What is $P(3 \leq X \leq 7)$?
2. Why can't a PDF dip below zero, even briefly?

### Intermediate

3. Can a PDF take values **greater than 1**? Consider a uniform distribution on $[0, 0.5]$ — what must its height be?
4. For the uniform $[0, 5]$ call center, find $t$ such that $P(X \leq t) = 0.9$.

### Advanced

5. Verify by simulation that for the uniform $[0, 0.5]$ distribution (density 2!), interval probabilities still come out correctly: estimate $P(0.1 \leq X \leq 0.3)$ and compare with the area $0.2 \times 2 = 0.4$.


## 7. Solutions


In [ ]:
# Exercise 1
print("1.  P(3<=X<=7) = 4 * 1/10 =", 4 * 1 / 10)

# Exercise 2
print("2.  Areas over that region would be negative probabilities — impossible.")

# Exercise 3
print("3.  Yes. Uniform on [0, 0.5] needs height 2 so that 0.5 * 2 = 1. "
      "Density is not probability; only AREAS are probabilities (and stay <= 1).")

# Exercise 4
print("4.  t * 1/5 = 0.9  ->  t = 4.5 minutes.")

# Exercise 5
x = rng.uniform(0, 0.5, size=500_000)
est = ((x >= 0.1) & (x <= 0.3)).mean()
print(f"5.  Simulated: {est:.4f}, area: 0.2 * 2 = 0.4 ✓")


## 8. Conclusion

- The **PDF** $f_X$ encodes a continuous distribution: $P(a \leq X \leq b)$ = area under $f$ between $a$ and $b$; single points are line segments with zero area.
- Density is the *rate of probability accumulation* — heights can even exceed 1, as long as areas behave.
- Three requirements: defined on all of $\mathbb{R}$, non-negative everywhere, total area 1.
- PMF ↔ heights and sums; PDF ↔ areas and integrals — the same probability rules, in continuous form.
